# Inspect GraphCast → CorrDiff Zarr

Loads the Zarr store produced by `pipeline.py` and summarizes:
- dimensions, coordinates, data variables, global attrs
- `era5` / `wrf` groups: shape, dtype, channel lists (with variable name + pressure)
- time axis cadence
- per-channel normalization (`*_center`, `*_scale`) values
- a couple of sample 2-D fields at a chosen time index
- **verification**: output Zarr vs source GraphCast NetCDF and vs base WRF Zarr

Both `era5` and `wrf` are 4-D arrays indexed as `(time, channel, south_north, west_east)`, time = 6-hourly from the GraphCast rollout.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path

# ---- paths used throughout the notebook ----
ZARR_PATH      = "/raid/apaudel/AURORA-CORRDIFF-ARTIFACTS/DATASETS/GRAPHCAST/ZARR_FOR_CORRDIFF/graphcast_2014_2015.zarr"
BASE_ZARR_PATH = "/raid/apaudel/AURORA-CORRDIFF-ARTIFACTS/DATASETS/CORRDIFF/TRAINING/hampton_2007_2015_wpd_avg.zarr"
GRAPHCAST_NC_DIR = "/raid/apaudel/AURORA-CORRDIFF-ARTIFACTS/DATASETS/GRAPHCAST/RAW_OUTPUTS/2014-2015"

ds = xr.open_zarr(ZARR_PATH, consolidated=False)
ds

## 1. Top-level summary

In [ ]:
print("Dimensions:")
for d, n in ds.sizes.items():
    print(f"  {d:20s} = {n}")

print("\nCoordinates:")
for c in ds.coords:
    v = ds.coords[c]
    print(f"  {c:20s} dims={v.dims} dtype={v.dtype} shape={v.shape}")

print("\nData variables:")
for v in ds.data_vars:
    da = ds[v]
    print(f"  {v:20s} dims={da.dims} dtype={da.dtype} shape={da.shape}")

print("\nGlobal attrs:")
for k, v in ds.attrs.items():
    print(f"  {k}: {v}")

## 2. Time axis

Check the 6-hourly cadence. `unique dt(h)` should ideally be `[6]`; other values mean there are gaps (e.g. between independent forecast initializations).

In [ ]:
t = ds["time"]
print("time attrs:", dict(t.attrs))

units = t.attrs.get("units", "")
if units.startswith("hours since "):
    origin = np.datetime64(units[len("hours since "):].strip(), "ns")
    times = origin + t.values.astype("timedelta64[h]")
else:
    times = t.values   # already datetime64 (decoded by xarray on read)

print(f"n_timesteps : {len(times)}")
print(f"first       : {times[0]}")
print(f"last        : {times[-1]}")
if len(times) > 1:
    dts = np.diff(times).astype('timedelta64[h]').astype(int)
    u, c = np.unique(dts, return_counts=True)
    print(f"unique dt(h): {dict(zip(u.tolist(), c.tolist()))}  # {{hours: count_of_gaps}}")

## 3. ERA5 and WRF group summaries

Channel names come from the coord `*_variable` (string name) and `*_pressure` (hPa; 0 or NaN for surface).

In [ ]:
def summarize(name):
    da = ds[name]
    print(f"=== {name} ===")
    print(f"  dims  : {da.dims}")
    print(f"  shape : {da.shape}")
    print(f"  dtype : {da.dtype}")

    # look up channel label coords (e.g. 'era5_variable', 'era5_pressure')
    var_coord  = f"{name}_variable"
    pres_coord = f"{name}_pressure"
    vnames = ds[var_coord].values  if var_coord  in ds.coords else None
    press  = ds[pres_coord].values if pres_coord in ds.coords else None
    n_ch = da.shape[1]

    print(f"  channels ({n_ch}):")
    print(f"    {'idx':>3s}  {'variable':<30s} {'pressure':>10s}")
    for i in range(n_ch):
        vn = vnames[i] if vnames is not None else ''
        pv = press[i]  if press  is not None else ''
        pv_str = '' if (pv == '' or (isinstance(pv, float) and np.isnan(pv))) else str(pv)
        print(f"    {i:3d}  {str(vn):<30s} {pv_str:>10s}")
    print()

if "era5" in ds: summarize("era5")
if "wrf"  in ds: summarize("wrf")

## 4. Validity masks

In [ ]:
for v in ("era5_valid", "wrf_valid"):
    if v in ds:
        da = ds[v]
        vals = da.values
        print(f"{v:12s} shape={da.shape} true={vals.sum()} false={(~vals).sum()}")

## 5. Normalization statistics (per channel)

Full listing of `*_center` and `*_scale` alongside the channel's variable/pressure label.

In [ ]:
def print_norm(group):
    ctr_name, scl_name = f"{group}_center", f"{group}_scale"
    if ctr_name not in ds or scl_name not in ds:
        return
    ctr = ds[ctr_name].values
    scl = ds[scl_name].values
    vnames = ds[f"{group}_variable"].values if f"{group}_variable" in ds.coords else [''] * len(ctr)
    press  = ds[f"{group}_pressure"].values if f"{group}_pressure" in ds.coords else [''] * len(ctr)

    print(f"=== {group}: center & scale ({len(ctr)} channels) ===")
    print(f"  {'idx':>3s}  {'variable':<30s} {'pressure':>10s}   {'center':>14s}   {'scale':>14s}")
    for i in range(len(ctr)):
        pv = press[i]
        pv_str = '' if (pv == '' or (isinstance(pv, float) and np.isnan(pv))) else str(pv)
        print(f"  {i:3d}  {str(vnames[i]):<30s} {pv_str:>10s}   {ctr[i]:>14.6g}   {scl[i]:>14.6g}")
    print()

print_norm("era5")
print_norm("wrf")

## 6. Plot a few fields at a selected time index

In [ ]:
TIME_INDEX = 0                  # pick any index in [0, n_timesteps)
ERA5_CHANNELS_TO_PLOT = [0, 1]  # by index
WRF_CHANNELS_TO_PLOT  = [0]

sel_time = times[TIME_INDEX]
print(f"Selected time index {TIME_INDEX}: {sel_time}")

def plot_channel(group, ch_idx):
    da = ds[group].isel(time=TIME_INDEX, **{f"{group}_channel": ch_idx})
    arr = da.values
    vn = ds[f"{group}_variable"].values[ch_idx] if f"{group}_variable" in ds.coords else ch_idx
    pv = ds[f"{group}_pressure"].values[ch_idx] if f"{group}_pressure" in ds.coords else ''
    label = f"{vn} p={pv}" if pv not in ('', None) and not (isinstance(pv, float) and np.isnan(pv)) else f"{vn}"
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(arr, origin="lower", cmap="viridis")
    ax.set_title(f"{group}[{ch_idx}] = {label}  @  {sel_time}")
    plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()
    print(f"  shape={arr.shape} min={arr.min():.4g} max={arr.max():.4g} mean={arr.mean():.4g}")

for i in ERA5_CHANNELS_TO_PLOT:
    plot_channel("era5", i)
for i in WRF_CHANNELS_TO_PLOT:
    plot_channel("wrf", i)

## 7. Verification — did the Zarr get packaged correctly?

Two independent checks:

### 7a. WRF round-trip
If `wrf.use_real=true`, the output Zarr's `wrf` field is simply rows copied from the base Zarr for the times present in GraphCast. So pixel values must match **exactly** (no regridding, same grid). We pick one WRF channel at one time, compare to the base Zarr at the same timestamp.

### 7b. ERA5 (GraphCast source)
The output `era5` block was produced by regridding the raw GraphCast NetCDF onto the WRF grid. We can't compare pixels byte-for-byte (grids differ), but we can sanity-check that:
- the same timestamp exists in the source `.nc`,
- min/max/mean of the regridded field are within the bounds of the raw field (values shouldn't explode).

A full pixel match would require re-running the same xESMF regrid on the raw file — we skip that here and just do the statistical bracket check.

In [ ]:
# ---------- 7a. WRF round-trip ----------
WRF_CH_IDX  = 0          # channel to test
T_IDX       = 0          # time index in the output Zarr

base = xr.open_zarr(BASE_ZARR_PATH, consolidated=False)

# decode time coords on both sides to datetime64[ns]
def _times_ns(da):
    v = da.values
    if np.issubdtype(v.dtype, np.datetime64):
        return v.astype('datetime64[ns]')
    units = da.attrs.get('units', '')
    if units.startswith('hours since '):
        origin = np.datetime64(units[len('hours since '):].strip(), 'ns')
        return (origin + v.astype('timedelta64[h]')).astype('datetime64[ns]')
    return v

out_times  = _times_ns(ds['time'])
base_times = _times_ns(base['wrf']['time'])
target_t   = out_times[T_IDX]
base_idx   = int(np.where(base_times == target_t)[0][0])

out_slice  = ds['wrf'].isel(time=T_IDX,  wrf_channel=WRF_CH_IDX).values
base_slice = base['wrf'].isel(time=base_idx, wrf_channel=WRF_CH_IDX).values

same = np.array_equal(out_slice, base_slice)
diff = np.abs(out_slice.astype('f8') - base_slice.astype('f8'))

ch_name = ds['wrf_variable'].values[WRF_CH_IDX] if 'wrf_variable' in ds.coords else WRF_CH_IDX
print(f"WRF channel {WRF_CH_IDX} ({ch_name}) @ {target_t}")
print(f"  output  shape={out_slice.shape}  min={out_slice.min():.4g}  max={out_slice.max():.4g}")
print(f"  base    shape={base_slice.shape} min={base_slice.min():.4g} max={base_slice.max():.4g}")
print(f"  array_equal : {same}")
print(f"  |diff| max  : {diff.max():.6g}")
print(f"  |diff| mean : {diff.mean():.6g}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, arr, title in zip(axes,
                          [out_slice, base_slice, diff],
                          [f"output zarr: wrf[{WRF_CH_IDX}]",
                           f"base zarr:   wrf[{WRF_CH_IDX}]",
                           "|output - base|"]):
    im = ax.imshow(arr, origin='lower', cmap='viridis' if 'diff' not in title else 'magma')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

base.close()

In [ ]:
# ---------- 7b. ERA5 bracket-check against raw GraphCast NetCDF ----------
# Pick an era5 channel that maps to a known GraphCast var.
# ERA5_VAR_NAME must be the short name stored in era5_variable coord.
# The corresponding GraphCast NetCDF variable name is read from GC_VAR_NAME.
ERA5_VAR_NAME = 't2m'
GC_VAR_NAME   = '2m_temperature'
ERA5_PRESSURE = 0                 # surface -> 0 (or NaN); for pressure levels set hPa value

# find the matching channel index in the output Zarr
vnames = ds['era5_variable'].values
press  = ds['era5_pressure'].values
mask = (vnames == ERA5_VAR_NAME)
ch_idx = int(np.where(mask)[0][0])
print(f"Output era5 channel {ch_idx}: variable={vnames[ch_idx]} pressure={press[ch_idx]}")

T_IDX = 0
target_t = out_times[T_IDX]
print(f"Target time: {target_t}")

# locate a GraphCast nc that contains this timestamp
nc_files = sorted(p for p in Path(GRAPHCAST_NC_DIR).rglob('*.nc')
                  if not p.name.startswith('ground_truth_'))
print(f"Scanning {len(nc_files)} GraphCast NetCDFs for time {target_t} ...")

src_ds = None
for f in nc_files:
    tmp = xr.open_dataset(f)
    tvals = np.asarray(tmp['time'].values).astype('datetime64[ns]')
    if target_t in tvals:
        src_ds = tmp.squeeze(drop=True).sel(time=target_t)
        print(f"  found in: {f.name}")
        break
    tmp.close()

if src_ds is None:
    print('  No matching NetCDF found. Skipping ERA5 bracket check.')
else:
    raw = src_ds[GC_VAR_NAME].values
    out_era5 = ds['era5'].isel(time=T_IDX, era5_channel=ch_idx).values
    print(f"  raw graphcast {GC_VAR_NAME}: shape={raw.shape} min={raw.min():.4g} max={raw.max():.4g} mean={raw.mean():.4g}")
    print(f"  zarr era5 [{ch_idx}]       : shape={out_era5.shape} min={out_era5.min():.4g} max={out_era5.max():.4g} mean={out_era5.mean():.4g}")

    within = (out_era5.min() >= raw.min() - 1e-3) and (out_era5.max() <= raw.max() + 1e-3)
    print(f"  regridded values within raw min/max bracket (bilinear should be): {within}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    im0 = axes[0].imshow(raw,      origin='lower', cmap='viridis')
    axes[0].set_title(f"raw GraphCast: {GC_VAR_NAME}\n(lat/lon grid)")
    plt.colorbar(im0, ax=axes[0])
    im1 = axes[1].imshow(out_era5, origin='lower', cmap='viridis')
    axes[1].set_title(f"output Zarr: era5[{ch_idx}] = {ERA5_VAR_NAME}\n(WRF grid, regridded)")
    plt.colorbar(im1, ax=axes[1])
    plt.tight_layout(); plt.show()

    src_ds.close()